# Halim Toddler — Colab **MAX POWER** (T4)

**Runtime → T4 GPU** then run cells 1→4.

| Setting | Value |
|---------|--------|
| Batch | **16** (sweet spot on T4) |
| Steps | **~3014** |
| GPU target | **~7–9 GB** / 15 GB |
| Est. time | **~4–5 h** |

Batch 24 uses VRAM but **~15s/step** → slower than batch 8. Use 16.
Mac prep: `./scripts/halim_colab_ready.sh` → upload **`halim_sft.zip`** in Cell 2.

In [ ]:
# Cell 1 — Mount Drive (output zip only)
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/Halim'
os.makedirs(WORK, exist_ok=True)
os.environ['HALIM_WORK'] = WORK
print('Drive:', WORK)
!ls -la "$WORK"

In [ ]:
# Cell 2 — Upload halim_sft.zip + setup
!pip install -q transformers peft trl datasets accelerate bitsandbytes

from google.colab import files
import os, shutil, zipfile
from pathlib import Path

for stale in ('sft', 'toddler_v1', 'train_toddler_colab.py', 'colab_drive_setup.py'):
    p = Path('/content') / stale
    if p.is_dir():
        shutil.rmtree(p)
    elif p.is_file():
        p.unlink()

print('Upload halim_sft.zip from Mac (~/Downloads/tradingbot/)')
uploaded = files.upload()
sft_zip = None
for name, data in uploaded.items():
    dest = Path('/content') / name
    dest.write_bytes(data)
    if name.endswith('.zip'):
        sft_zip = dest

if sft_zip is None:
    sft_zip = next(Path('/content').glob('halim_sft*.zip'), None)
if sft_zip is None:
    raise FileNotFoundError('Upload halim_sft.zip')

with zipfile.ZipFile(sft_zip, 'r') as zf:
    zf.extractall('/content')

%cd /content
!python colab_drive_setup.py

In [ ]:
# Cell 3 — Fast train (batch 16 sweet spot — NOT 24)
%cd /content
import os, shutil
from pathlib import Path

adapter = Path('toddler_v1/lora_adapter')
if adapter.exists():
    shutil.rmtree(adapter)

os.environ['HALIM_OUT_DIR'] = '/content/toddler_v1'
os.environ['HALIM_FRESH_TRAIN'] = 'true'
os.environ['HALIM_CONTINUE_LORA'] = 'false'
os.environ['HALIM_FAST_PATH'] = 'auto'
os.environ['HALIM_MAX_POWER'] = 'true'
os.environ['HALIM_BATCH_SIZE'] = '16'
os.environ['HALIM_GRAD_ACCUM'] = '1'
os.environ['HALIM_FP16'] = 'false'
os.environ['HALIM_BF16'] = 'false'

!python train_toddler_colab.py

In [ ]:
# Cell 3b — ONLY if Cell 3 OOM: batch 12 retry
%cd /content
import os, shutil
from pathlib import Path

adapter = Path('toddler_v1/lora_adapter')
if adapter.exists():
    shutil.rmtree(adapter)

os.environ['HALIM_OUT_DIR'] = '/content/toddler_v1'
os.environ['HALIM_FRESH_TRAIN'] = 'true'
os.environ['HALIM_MAX_POWER'] = 'false'
os.environ['HALIM_BATCH_SIZE'] = '12'
os.environ['HALIM_GRAD_ACCUM'] = '1'
os.environ['HALIM_FP16'] = 'false'
os.environ['HALIM_BF16'] = 'false'

!python train_toddler_colab.py

In [ ]:
# Cell 4 — Zip to Drive + Mac install
import json, shutil, subprocess
from pathlib import Path

WORK = Path('/content/drive/MyDrive/Halim')
state = json.loads((WORK / 'halim_colab_state.json').read_text()) if (WORK / 'halim_colab_state.json').is_file() else {}
out_name = state.get('next_output_zip', 'halim_toddler_v4.zip')

src = Path('/content/toddler_v1')
if not (src / 'merged').is_dir():
    raise FileNotFoundError('Wait for Cell 3 to finish (merged/ missing)')

shutil.copytree(src, WORK / 'toddler_v1', dirs_exist_ok=True)
subprocess.run(['zip', '-r', out_name, 'toddler_v1'], cwd=str(WORK), check=True)
print('Saved:', WORK / out_name)
print('Mac: ./scripts/halim_apply_colab_checkpoint.sh && ./scripts/halim_record_train.sh')